<a href="https://colab.research.google.com/github/thomas-sutter/ai-aml-mvp2value/blob/main/1_exploratory_data_analysis_blueprint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IBM Synthetic Data — Core Banking & AML — **EDA Blueprint** (Iterative / Non‑Sequential)

> **Purpose.** Professional, reproducible EDA for the IBM Synthetic Core Banking & AML dataset (CSV).  
> **Flow.** *Discovering → Joining → Validating → Structuring → Cleaning → Presenting* (iterate, don't treat as a strict sequence).  
> **Focus.** AML-relevant patterns (fan‑in / fan‑out / short cycles), label coherence, data quality, and next‑step recommendations.

**Notebook Map**  
- 0️⃣ `0_Setup` (installs, imports, config, load CSVs)  
- 1️⃣ `1_Discovering` (overview profiles & first visuals)  
- 2️⃣ `2_Joining` (enrich xfers with accounts/banks/b2b)  
- 3️⃣ `3_Validating` (balances, FX, labels, rules)  
- 4️⃣ `4_Structuring` (windowed features & network cues)  
- 5️⃣ `5_Cleaning` (documented cleaning ops)  
- 6️⃣ `6_Presenting` (Sankey, network mini‑multiples, cohorts)  
- 7️⃣ `7_Findings_and_Next_Steps` (actionable summary)


## 0_Setup


In [ ]:
# Optional: uncomment any of these if not available in your Colab/runtime.
# !pip install --quiet pyarrow polars plotly networkx

import os, sys, math, json, textwrap, itertools, collections, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Try optional libs
try:
    import plotly.graph_objects as go
except Exception:
    go = None

try:
    import networkx as nx
except Exception:
    nx = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
warnings.filterwarnings("ignore")

# ---- Paths (edit as needed) ----
DATA_PATH_CANDIDATES = {
    "bank_xfers": [
        "/content/sd175_Small_bank_xfers.csv",
        "/mnt/data/sd175_Small_bank_xfers.csv",
        "./sd175_Small_bank_xfers.csv"
    ],
    "accts_people": [
        "/content/sd175_Small_liquid_accts_people.csv",
        "/mnt/data/sd175_Small_liquid_accts_people.csv",
        "./sd175_Small_liquid_accts_people.csv"
    ],
    "accts_companies": [
        "/content/sd175_Small_liquid_accts_companies.csv",
        "/mnt/data/sd175_Small_liquid_accts_companies.csv",
        "./sd175_Small_liquid_accts_companies.csv"
    ],
    "b2b": [
        "/content/sd175_Small_b2b.csv",
        "/mnt/data/sd175_Small_b2b.csv",
        "./sd175_Small_b2b.csv"
    ],
    "banks": [
        "/content/sd175_Small_banks.csv",
        "/mnt/data/sd175_Small_banks.csv",
        "./sd175_Small_banks.csv"
    ]
}

def pick_first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

PATHS = {k: pick_first_existing(v) for k,v in DATA_PATH_CANDIDATES.items()}
print("Resolved paths:", json.dumps(PATHS, indent=2))

# ---- Loaders ----
def read_csv_smart(path, parse_ts_cols=None, dtype=None):
    if path is None:
        return None
    parse_ts_cols = parse_ts_cols or []
    try:
        df = pd.read_csv(path, dtype=dtype, parse_dates=parse_ts_cols, low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(path, dtype=dtype, parse_dates=parse_ts_cols, encoding="latin-1", low_memory=False)
    return df

bank_xfers = read_csv_smart(PATHS["bank_xfers"], parse_ts_cols=["Timestamp"])
accts_people = read_csv_smart(PATHS["accts_people"])
accts_companies = read_csv_smart(PATHS["accts_companies"])
b2b = read_csv_smart(PATHS["b2b"])
banks = read_csv_smart(PATHS["banks"])

def data_overview(df, name):
    if df is None:
        print(f"[WARN] {name}: not loaded (path missing).")
        return
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print("dtypes:
", df.dtypes.head(20))
    print("head:
", df.head(2))
    print()

for name, df in {
    "bank_xfers": bank_xfers,
    "accts_people": accts_people,
    "accts_companies": accts_companies,
    "b2b": b2b,
    "banks": banks
}.items():
    data_overview(df, name)

# ---- Data Dictionary utility ----
def build_data_dictionary(dfs):
    rows = []
    for name, df in dfs.items():
        if df is None:
            continue
        for col in df.columns:
            s = df[col]
            na_rate = float(s.isna().mean())
            nunique = int(s.nunique(dropna=True))
            sample_vals = ", ".join([str(v) for v in s.dropna().unique()[:5]])
            rows.append({
                "table": name,
                "column": col,
                "dtype": str(s.dtype),
                "missing_rate": round(na_rate, 6),
                "nunique": nunique,
                "examples": sample_vals
            })
    dd = pd.DataFrame(rows).sort_values(["table", "column"]).reset_index(drop=True)
    return dd

data_dict = build_data_dictionary({
    "bank_xfers": bank_xfers,
    "accts_people": accts_people,
    "accts_companies": accts_companies,
    "b2b": b2b,
    "banks": banks
})
data_dict.head(20)


## 1_Discovering — First Look & Profiles


In [ ]:
# --- Basic temporal and label distributions (bank_xfers) ---
if bank_xfers is not None and not bank_xfers.empty:
    bx = bank_xfers.copy()
    # Ensure Timestamp is datetime
    if "Timestamp" in bx.columns and not np.issubdtype(bx["Timestamp"].dtype, np.datetime64):
        bx["Timestamp"] = pd.to_datetime(bx["Timestamp"], errors="coerce")
    # Amounts
    amt_cols = [c for c in ["Amount_Paid","Amount_Received"] if c in bx.columns]
    print("Amount columns:", amt_cols)

    # Time range
    if "Timestamp" in bx.columns:
        print("Time span:", bx["Timestamp"].min(), "→", bx["Timestamp"].max())

    # Label distributions
    for lab in ["Is_Laundering", "Is_Cheque_Fraud", "Is_APP_Fraud", "Laundering_Type", "Transaction_Type"]:
        if lab in bx.columns:
            print(f"--- {lab} ---")
            print(bx[lab].value_counts(dropna=False).head(20))

    # Plots (matplotlib, no seaborn)
    fig = plt.figure(figsize=(6,4))
    if "Amount_Paid" in bx.columns:
        bx["Amount_Paid"].plot(kind="hist", bins=60)
        plt.title("Histogram: Amount_Paid")
        plt.xlabel("Amount_Paid")
    plt.show()

    fig = plt.figure(figsize=(6,4))
    if "Amount_Received" in bx.columns:
        bx["Amount_Received"].plot(kind="hist", bins=60)
        plt.title("Histogram: Amount_Received")
        plt.xlabel("Amount_Received")
    plt.show()

    # Time-of-day heatmap (weekday x hour)
    if "Timestamp" in bx.columns:
        tmp = bx.copy()
        tmp["weekday"] = tmp["Timestamp"].dt.weekday
        tmp["hour"] = tmp["Timestamp"].dt.hour
        pivot = tmp.pivot_table(index="weekday", columns="hour", values="From_Account", aggfunc="count").fillna(0)
        plt.figure(figsize=(10,3.8))
        plt.imshow(pivot.values, aspect="auto", interpolation="nearest")
        plt.title("Count of transactions — Weekday x Hour")
        plt.xlabel("Hour"); plt.ylabel("Weekday (0=Mon)")
        plt.colorbar()
        plt.show()

    # Top 20 Transaction_Type
    if "Transaction_Type" in bx.columns:
        top_types = bx["Transaction_Type"].value_counts().head(20)
        top_types.plot(kind="bar", figsize=(10,3.2))
        plt.title("Top 20 Transaction_Type (count)")
        plt.tight_layout(); plt.show()


## 2_Joining — Create `xfers_enriched` with account, bank, and B2B context


In [ ]:
def _prefix_join(df_left, df_right, left_key, right_key, prefix):
    # Helper: left join and add prefix to joined columns
    cols_right = [c for c in df_right.columns if c != right_key]
    renamed = {c: f"{prefix}{c}" for c in cols_right}
    return df_left.merge(df_right, how="left", left_on=left_key, right_on=right_key).rename(columns=renamed).drop(columns=[right_key])

def build_xfers_enriched(bank_xfers, accts_people, accts_companies, banks, b2b):
    if bank_xfers is None:
        return None, {}
    x = bank_xfers.copy()
    join_report = {}

    # From_Account & To_Account → Accounts (people + companies concatenated)
    accounts = None
    if accts_people is not None:
        accounts = accts_people.copy()
    if accts_companies is not None:
        accounts = pd.concat([accounts, accts_companies], axis=0) if accounts is not None else accts_companies.copy()

    if accounts is not None and "Account_ID" in accounts.columns:
        before = len(x)
        # From side
        x = _prefix_join(x, accounts, left_key="From_Account", right_key="Account_ID", prefix="from_")
        # To side
        x = _prefix_join(x, accounts, left_key="To_Account", right_key="Account_ID", prefix="to_")
        join_report["accounts_join_rows"] = {"before": before, "after": len(x)}

    # Banks metadata
    if banks is not None and "Bank_ID" in banks.columns:
        # map From_Bank
        x = _prefix_join(x, banks, left_key="From_Bank", right_key="Bank_ID", prefix="from_bank_")
        # map To_Bank
        x = _prefix_join(x, banks, left_key="To_Bank", right_key="Bank_ID", prefix="to_bank_")

    # B2B overlay (optional): link company entities if IDs line up
    if b2b is not None and "Company_ID" in b2b.columns and "from_Entity_ID" in x.columns:
        x = x.merge(b2b.add_prefix("from_company_"), how="left", left_on="from_Entity_ID", right_on="from_company_Company_ID")
    if b2b is not None and "Company_ID" in b2b.columns and "to_Entity_ID" in x.columns:
        x = x.merge(b2b.add_prefix("to_company_"), how="left", left_on="to_Entity_ID", right_on="to_company_Company_ID")

    return x, join_report

xfers_enriched, join_report = build_xfers_enriched(bank_xfers, accts_people, accts_companies, banks, b2b)
print("Join report:", join_report)
xfers_enriched.head(3) if xfers_enriched is not None else None


## 3_Validating — Balances, FX, Label Coherence, Rule Checks


In [ ]:
def validate_balances(df, tol=1e-6):
    issues = []
    if df is None or df.empty:
        return pd.DataFrame(issues)

    need = {"From_Initial_Balance","From_End_Balance","Amount_Paid","To_Initial_Balance","To_End_Balance","Amount_Received"}
    if not need.issubset(df.columns):
        return pd.DataFrame(issues)

    exp_from_end = df["From_Initial_Balance"] - df["Amount_Paid"]
    exp_to_end = df["To_Initial_Balance"] + df["Amount_Received"]
    bad_from = (df["From_End_Balance"] - exp_from_end).abs() > tol
    bad_to = (df["To_End_Balance"] - exp_to_end).abs() > tol

    if bad_from.any():
        issues.append({"issue":"from_balance_mismatch","rows":int(bad_from.sum()),"rate":float(bad_from.mean())})
    if bad_to.any():
        issues.append({"issue":"to_balance_mismatch","rows":int(bad_to.sum()),"rate":float(bad_to.mean())})

    return pd.DataFrame(issues)

def validate_fx(df, z_thresh=4.0):
    issues = []
    if df is None or df.empty:
        return pd.DataFrame(issues)
    need = {"Payment_Currency","Receiving_Currency","Amount_Paid","Amount_Received"}
    if not need.issubset(df.columns):
        return pd.DataFrame(issues)

    mask = (df["Payment_Currency"].notna() & df["Receiving_Currency"].notna() & (df["Amount_Paid"]>0))
    sub = df.loc[mask].copy()
    if sub.empty:
        return pd.DataFrame(issues)

    sub["implied_fx"] = sub["Amount_Received"] / sub["Amount_Paid"]
    # Compute z-score per currency pair
    sub["pair"] = sub["Payment_Currency"].astype(str) + "→" + sub["Receiving_Currency"].astype(str)
    out_rows = []
    for pair, grp in sub.groupby("pair"):
        m = grp["implied_fx"].mean()
        s = grp["implied_fx"].std(ddof=0)
        if s == 0 or np.isnan(s):
            continue
        z = (grp["implied_fx"] - m) / s
        outliers = (z.abs() > z_thresh)
        if outliers.any():
            out_rows.append({"pair": pair, "outlier_rows": int(outliers.sum()), "rate": float(outliers.mean())})
    return pd.DataFrame(out_rows)

def validate_labels(df):
    issues = []
    if df is None or df.empty:
        return pd.DataFrame(issues)
    if "Is_Laundering" in df.columns and "Laundering_Type" in df.columns:
        bad = (df["Is_Laundering"]==1) & (df["Laundering_Type"].isna() | (df["Laundering_Type"].astype(str).str.lower().isin(["", "none", "clean"])))
        if bad.any():
            issues.append({"issue":"laundering_label_without_type","rows":int(bad.sum()),"rate":float(bad.mean())})
    return pd.DataFrame(issues)

def validate_rules(df):
    issues = []
    if df is None or df.empty:
        return pd.DataFrame(issues)
    for cols, name in [
        (["Sufficient_Funds","Overdraft_Okay"], "insufficient_and_no_overdraft"),
        (["Is_All_Cash","Payment_Format"], "cash_vs_payment_format"),
        (["Is_Hold"], "holds_present")
    ]:
        for c in cols:
            if c not in df.columns:
                break
        # Example: count insufficient funds with overdraft not ok but transaction succeeded (heuristic)
    if set(["Sufficient_Funds","Overdraft_Okay"]).issubset(df.columns):
        bad = (df["Sufficient_Funds"]==0) & (df["Overdraft_Okay"]==0)
        issues.append({"issue":"insufficient_funds_and_no_overdraft", "rows": int(bad.sum()), "rate": float(bad.mean())})
    if "Is_All_Cash" in df.columns and "Payment_Format" in df.columns:
        bad = (df["Is_All_Cash"]==1) & (~df["Payment_Format"].astype(str).str.lower().str.contains("cash"))
        issues.append({"issue":"all_cash_but_format_not_cash", "rows": int(bad.sum()), "rate": float(bad.mean())})
    if "Is_Hold" in df.columns:
        holds = (df["Is_Hold"]==1)
        issues.append({"issue":"holds_flagged", "rows": int(holds.sum()), "rate": float(holds.mean())})
    return pd.DataFrame(issues)

def build_issue_log(df):
    parts = []
    for fn in [validate_balances, validate_fx, validate_labels, validate_rules]:
        try:
            res = fn(df)
            if res is not None and not res.empty:
                parts.append(res)
        except Exception as e:
            parts.append(pd.DataFrame([{"issue":f"{fn.__name__}_error", "rows":-1, "rate":-1.0, "error":str(e)}]))
    if parts:
        out = pd.concat(parts, ignore_index=True)
        out = out.sort_values(by=["rows"], ascending=False)
    else:
        out = pd.DataFrame(columns=["issue","rows","rate"])
    return out

issue_log = build_issue_log(bank_xfers if bank_xfers is not None else xfers_enriched)
issue_log.head(20)


## 4_Structuring — Windowed Features & Network Signals


In [ ]:
def make_long_edges(df):
    # Convert transfers into a long 'account-centric' table for windowed rollups
    need = ["Timestamp","From_Account","To_Account","Amount_Paid","Amount_Received"]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"Missing columns: {miss}")
    x = df.copy()
    x["Timestamp"] = pd.to_datetime(x["Timestamp"], errors="coerce")
    # Outgoing leg
    out = x[["Timestamp","From_Account","To_Account","Amount_Paid"]].rename(
        columns={"From_Account":"account_id","To_Account":"counterparty_id","Amount_Paid":"amount"})
    out["direction"] = "out"
    # Incoming leg
    inc = x[["Timestamp","To_Account","From_Account","Amount_Received"]].rename(
        columns={"To_Account":"account_id","From_Account":"counterparty_id","Amount_Received":"amount"})
    inc["direction"] = "in"
    long_df = pd.concat([out, inc], ignore_index=True)
    long_df = long_df.sort_values("Timestamp")
    return long_df

def rolling_features(long_df, window="7D"):
    if long_df is None or long_df.empty:
        return pd.DataFrame()
    long_df = long_df.dropna(subset=["Timestamp"])
    long_df = long_df.sort_values("Timestamp")
    # Group by account_id & direction
    feats = []
    for (acc, direction), g in long_df.groupby(["account_id","direction"], sort=False):
        g = g.set_index("Timestamp").sort_index()
        # Basic rolling sums and counts
        g["amount_sum"] = g["amount"].rolling(window=window).sum()
        g["tx_count"] = g["amount"].rolling(window=window).count()
        # Unique counterparties over window (approx via expanding set on day granularity)
        daily = g.resample("1D").agg({"amount":"sum","counterparty_id": lambda s: s.nunique()})
        daily["burstiness"] = daily["amount"].rolling(window=7).max() / (daily["amount"].rolling(window=7).mean()+1e-9)
        df_feat = pd.DataFrame({
            "account_id": acc,
            "direction": direction,
            "ts": daily.index,
            "sum_amount_1D": daily["amount"],
            "unique_cpty_1D": daily["counterparty_id"],
            "burstiness_7D": daily["burstiness"]
        })
        feats.append(df_feat.reset_index(drop=True))
    return pd.concat(feats, ignore_index=True) if feats else pd.DataFrame()

def fan_in_out_scores(long_df, ref_window="7D"):
    if long_df is None or long_df.empty:
        return pd.DataFrame()
    # Count unique counterparties per account over the whole dataset (approx static score)
    grp = long_df.groupby(["account_id","direction"])["counterparty_id"].nunique().unstack(fill_value=0)
    grp = grp.rename(columns={"in":"unique_in","out":"unique_out"})
    grp["fan_in_score"] = grp.get("unique_in", 0)
    grp["fan_out_score"] = grp.get("unique_out", 0)
    return grp.reset_index()

# Build
if bank_xfers is not None:
    long_df = make_long_edges(bank_xfers)
    roll7 = rolling_features(long_df, window="7D")
    fan_scores = fan_in_out_scores(long_df)
    print("roll7 shape:", roll7.shape, "fan_scores shape:", fan_scores.shape)
    fan_scores.sort_values("fan_out_score", ascending=False).head(10)


In [ ]:
def build_graph_for_sample(df, sample_n=3000):
    if nx is None:
        print("[INFO] networkx not available; skip graph.")
        return None
    if df is None or df.empty:
        return None
    cols = ["From_Account","To_Account","Amount_Paid","Amount_Received","Timestamp","Is_Laundering","Laundering_Type"]
    cols = [c for c in cols if c in df.columns]
    sample = df[cols].sample(min(sample_n, len(df)), random_state=42)
    G = nx.DiGraph()
    for _, r in sample.iterrows():
        u = r.get("From_Account"); v = r.get("To_Account")
        if pd.isna(u) or pd.isna(v):
            continue
        G.add_edge(u, v, amount=float(r.get("Amount_Paid", 0)), ts=r.get("Timestamp", None),
                   is_laund=int(r.get("Is_Laundering", 0)) if not pd.isna(r.get("Is_Laundering", np.nan)) else 0,
                   ltype=str(r.get("Laundering_Type", "")))
    return G

def simple_cycle_counts(G, max_len=5, limit=5000):
    if G is None:
        return pd.DataFrame()
    cycles = []
    for i, cyc in enumerate(nx.simple_cycles(G)):
        if i>limit: break
        if 2 <= len(cyc) <= max_len:
            cycles.append(cyc)
    # Count participation per node
    counts = {}
    for cyc in cycles:
        for node in cyc:
            counts[node] = counts.get(node, 0) + 1
    out = pd.DataFrame([{"account_id":k, "cycle_count":v} for k,v in counts.items()]).sort_values("cycle_count", ascending=False)
    return out

G = build_graph_for_sample(bank_xfers, sample_n=5000) if bank_xfers is not None else None
cycle_rank = simple_cycle_counts(G, max_len=5, limit=3000) if G is not None else pd.DataFrame()
cycle_rank.head(10)


## 5_Cleaning — Documented Operations


In [ ]:
def clean_bank_xfers(df):
    x = df.copy()
    # Normalize string empties to NaN
    for c in x.select_dtypes(include=["object"]).columns:
        x[c] = x[c].replace({"": np.nan, "NA": np.nan, "None": np.nan})
    # Ensure Timestamp dtype
    if "Timestamp" in x.columns:
        x["Timestamp"] = pd.to_datetime(x["Timestamp"], errors="coerce")
    # Remove duplicates (conservative)
    x = x.drop_duplicates()
    # Optional: clip extreme amounts for EDA plots only (not for modeling!)
    if "Amount_Paid" in x.columns:
        q_hi = x["Amount_Paid"].quantile(0.999)
        x["_Amount_Paid_plot"] = x["Amount_Paid"].clip(upper=q_hi)
    if "Amount_Received" in x.columns:
        q_hi2 = x["Amount_Received"].quantile(0.999)
        x["_Amount_Received_plot"] = x["Amount_Received"].clip(upper=q_hi2)
    return x

bank_xfers_clean = clean_bank_xfers(bank_xfers) if bank_xfers is not None else None
bank_xfers_clean.head(3) if bank_xfers_clean is not None else None


## 6_Presenting — Sankey, Network Samples, Cohorts


In [ ]:
# Sankey From_Bank → To_Bank (volume)
def sankey_bank_to_bank(df, top_n=12):
    if go is None:
        print("[INFO] plotly not available; skip sankey.")
        return None
    need = {"From_Bank","To_Bank","Amount_Paid"}
    if not need.issubset(df.columns):
        print("[WARN] Missing columns for Sankey:", need - set(df.columns))
        return None
    agg = df.groupby(["From_Bank","To_Bank"])["Amount_Paid"].sum().reset_index()
    agg = agg.sort_values("Amount_Paid", ascending=False).head(top_n)
    labels = list(pd.unique(agg[["From_Bank","To_Bank"]].values.ravel("K")))
    label_idx = {lab:i for i, lab in enumerate(labels)}
    source = agg["From_Bank"].map(label_idx).tolist()
    target = agg["To_Bank"].map(label_idx).tolist()
    value = agg["Amount_Paid"].tolist()
    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=15, label=labels),
        link=dict(source=source, target=target, value=value)
    )])
    fig.update_layout(title_text="Top Bank→Bank Flows (by Amount_Paid)", font_size=10)
    return fig

if bank_xfers_clean is not None:
    fig = sankey_bank_to_bank(bank_xfers_clean, top_n=14)
    fig.show() if fig is not None else None


## 7_Findings_and_Next_Steps — Checklist


**Top Findings (fill as you iterate):**
- [ ] Data coverage: time range, currencies, formats
- [ ] Label distribution & imbalance quantified
- [ ] Balance / FX sanity checks summary
- [ ] Fan‑in / fan‑out hubs and top cycle participants
- [ ] Suspicious patterns (accounts, banks, countries)

**Concrete Recommendations:**
- [ ] Feature candidates (name, window, rationale) -> export as CSV (feature_catalog.csv)
- [ ] Evaluation plan (time‑split; PR‑AUC/F1; calibration)
- [ ] Governance notes (ground‑truth usage, bias checks, documentation)
